# Text representation benchmark — off-market first

Compare **no text**, the existing **47-term TF-IDF**, **47-term binary Bag-of-Words**, and **three full 1,000-term approaches** on the same chronological folds with the same model hyperparameters.

- Full-vocabulary terms are selected **only from earlier training remarks**, without accessing sale prices or validation text.
- The *same vocabulary* is used for all three full-vocabulary representations within each fold.
- Historical comparable-sale features are computed once per fold and reused across experiments.
- This notebook runs only on earlier months (default September–November 2025). It **never** tunes on December 2025–May 2026.
- The old 47-term model and its default factory path remain unchanged.
- The 47 terms were historically selected using Elastic Net/Lasso; a separate supervised selection audit is appropriate if their original selection used validation labels.

Run from the repo root. No new Python libraries are required beyond the existing model dependencies.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from models.text_features import TEXT_VARIANTS
from text_benchmark import run_benchmark

BACKEND = "lightgbm"     # or "xgboost"; test one library at a time
TRACK = "off_market"     # list_aware supported too
MONTHS = ["2025-09", "2025-10", "2025-11"]
VARIANTS = list(TEXT_VARIANTS)
OUTPUT_DIR = Path("outputs/text_benchmark")

# Paste your *tuned* parameter overrides here for the fairest comparison.
# Leaving this empty uses repository defaults, which were NOT your Optuna winners.
PARAMS = {}

# Alternative: specify a JSON file containing your tuned params:
PARAMS_FILE = None  # e.g. Path("tuning/best_lightgbm_off_market.json")
if PARAMS_FILE is not None:
    PARAMS = json.loads(Path(PARAMS_FILE).read_text(encoding="utf-8"))

print("Backend:", BACKEND, "| Track:", TRACK)
print("Variants:", VARIANTS)
print("Model overrides:", PARAMS if PARAMS else "Repository defaults (not tuned)")


In [ ]:
pooled, monthly, predictions = run_benchmark(
    backend=BACKEND,
    track=TRACK,
    months=MONTHS,
    variants=VARIANTS,
    params=PARAMS,
    output_dir=OUTPUT_DIR,
    center_target=True,
)
display(pooled)


In [ ]:
display(monthly.pivot(index="month", columns="variant", values="mdape").round(3))
display(monthly.pivot(index="month", columns="variant", values="mape").round(3))

baseline = pooled.loc[pooled.variant.eq("selected_tfidf"), "mdape"]
if len(baseline):
    comparison = pooled.copy()
    comparison["mdape_minus_47_tfidf_pp"] = comparison["mdape"] - baseline.iloc[0]
    display(comparison.sort_values("mdape").round(3))

print("Saved paired property predictions in", OUTPUT_DIR / f"{BACKEND}_{TRACK}_predictions.csv")


## Next step (after development results)

If a candidate consistently helps on **September–November 2025**, freeze its vocabulary rule and model parameters, and change only the feature name in the usual `model_run.ipynb` `MODEL_SPECS`:

```python
MODEL_SPECS = [
    (LightGBMRegressor, "Xpt_spatiotemporal__full_binary", YOUR_TUNED_PARAMS)
]
```

Supported suffixes: `no_text`, `selected_tfidf`, `selected_binary`, `full_tfidf`, `full_binary`, `full_count`.

Then perform **one** December 2025–May 2026 confirmation for the shortlisted representation. The standalone benchmark saves all earlier development predictions and setup/vocabulary information for reproducibility; it does not alter `logs.md`, the production factory, or any existing saved model.
